# 16 Final Results And Model Selection

This notebook is the **final thesis synthesis notebook** for hourly Dutch day-ahead electricity price forecasting.

It is not another screening notebook. Its job is to bring the phased evidence together in one place and answer the thesis-level questions:
- which candidate models and feature-set variants survived the earlier phases
- which final model is selected under a pre-declared validation rule
- how that selected model performs on the untouched test set
- why that model is chosen over the alternatives, in both statistical and practical terms

This notebook should be used only **after the methodology is frozen**. Test results must not be used to choose the winner.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.reporting import find_latest_run, load_csv, load_json

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
run_root = output_root / "runs"


def latest_run_matching(pattern: str) -> Path | None:
    matches = sorted(run_root.glob(pattern))
    return matches[-1] if matches else None


def load_run_bundle(run_dir: Path | None) -> dict[str, object] | None:
    if run_dir is None:
        return None
    bundle: dict[str, object] = {"run_dir": run_dir}
    csv_names = [
        "metrics_overall.csv",
        "metrics_by_lead_day.csv",
        "metrics_by_reporting_level.csv",
        "origin_timing_summary.csv",
        "diebold_mariano_by_reporting_level.csv",
    ]
    json_names = [
        "official_naive_reference.json",
        "run_summary.json",
    ]
    for name in csv_names:
        path = run_dir / name
        if path.exists():
            bundle[path.stem] = load_csv(run_dir, name)
    for name in json_names:
        path = run_dir / name
        if path.exists():
            bundle[path.stem] = load_json(run_dir, name)
    return bundle


print(output_root)


## 1. Role Of This Notebook

This notebook should sit **above** the phased notebooks.

The earlier notebooks remain the methodological trail:
- `00`: frozen setup and objective case-week logic
- `01-08`: benchmark stack through `FS2`
- `09-12`: phased `FS3` work
- `13-15`: future `FS4` methodology and design work

This notebook is different. It should not re-argue every intermediate step in full detail. Instead it should:
- restate the frozen comparison framework once
- load only the final candidate runs
- select the final model on validation only
- reveal and interpret the held-out test results afterward
- provide the thesis-ready conclusion and model recommendation


In [ ]:
FINAL_CANDIDATE_SOURCES = [
    {
        "name": "benchmark_stack",
        "source_type": "run_label",
        "source_value": "model_comparison",
        "required": True,
        "notes": "Final benchmark reference from FS0-FS2.",
    },
    {
        "name": "fs3_final",
        "source_type": "pattern",
        "source_value": "*_fs3_combo_promoted*",
        "required": False,
        "notes": "Promoted FS3 final run if FS3 is part of the final candidate set.",
    },
    {
        "name": "fs4_final",
        "source_type": "pattern",
        "source_value": "*_fs4_final*",
        "required": False,
        "notes": "Future promoted FS4 final run after the FS4 methodology is fully frozen and executed.",
    },
]


def resolve_candidate_source(spec: dict[str, object]) -> Path | None:
    if spec["source_type"] == "run_label":
        try:
            return find_latest_run(output_root, str(spec["source_value"]))
        except FileNotFoundError:
            return None
    if spec["source_type"] == "pattern":
        return latest_run_matching(str(spec["source_value"]))
    return None


candidate_source_table = pd.DataFrame(
    [
        {
            **spec,
            "resolved_run_dir": resolve_candidate_source(spec),
        }
        for spec in FINAL_CANDIDATE_SOURCES
    ]
)
candidate_source_table


## 2. Freeze Prerequisites

This notebook should not be treated as valid until the following are true:
- the split, horizon, forecast-origin rule, UTC handling, and leakage policy are frozen
- the official naive benchmark has already been selected on validation
- the benchmark stack is stable
- any `FS3` or `FS4` inclusion has already been promoted using validation only
- no candidate was added after looking at the final test results

If any of these conditions is false, this notebook becomes a source of leakage or post-hoc selection bias.


In [ ]:
FINAL_NOTEBOOK_FREEZE_CHECKLIST = {
    "benchmark_methodology_frozen": False,
    "official_naive_selected_on_validation": False,
    "final_candidate_set_frozen_before_test_review": False,
    "fs3_promotions_if_any_frozen_on_validation": False,
    "fs4_promotions_if_any_frozen_on_validation": False,
    "test_set_not_used_for_design": False,
}

pd.DataFrame(
    {
        "item": list(FINAL_NOTEBOOK_FREEZE_CHECKLIST.keys()),
        "ready": list(FINAL_NOTEBOOK_FREEZE_CHECKLIST.values()),
    }
)


## 3. Final Model-Selection Rule

This notebook should make the winner-selection rule explicit **before** showing the final test comparison.

Recommended rule for the current thesis objective:
- candidate set: only the final promoted models, not every rejected screening variant
- primary criterion: **validation MAE on `stitched_all_horizon`**
- tie-break 1: validation MAE on `d_only`
- tie-break 2: validation `rMAE` on `stitched_all_horizon`
- tie-break 3: Diebold-Mariano evidence against the official naive and the strongest simpler benchmark
- tie-break 4: runtime and practical practicality

Why this is the recommended primary rule:
- the business task is `D..D+4`, not only `D`
- `stitched_all_horizon` gives one operational summary across the full horizon
- `d_only` still matters as an operational secondary check because day `D` is the most immediate part of the forecast

Important:
- the final test set should be used only after the winner is frozen
- if the thesis later decides that `D` alone is the main optimization target, this rule must be changed **before** viewing test-based winner tables


In [ ]:
FINAL_SELECTION_RULE = {
    "candidate_scope": "final_promoted_models_only",
    "primary_split": "validation",
    "primary_reporting_level": "stitched_all_horizon",
    "primary_metric": "mae",
    "tie_breaks": [
        "validation_d_only_mae",
        "validation_stitched_all_horizon_rmae_vs_official_naive",
        "diebold_mariano_vs_official_naive_and_strongest_simpler_benchmark",
        "runtime_and_practicality",
    ],
    "test_usage": "final_held_out_reporting_only",
}

FINAL_DECISION_LOG_TEMPLATE = {
    "selected_model": None,
    "selected_fs_level": None,
    "selection_split": "validation",
    "selection_reporting_level": "stitched_all_horizon",
    "selection_metric": "mae",
    "why_selected": None,
    "closest_runner_up": None,
    "test_results_revealed_after_freeze": False,
}

pd.DataFrame(
    {
        "setting": list(FINAL_SELECTION_RULE.keys()),
        "value": list(FINAL_SELECTION_RULE.values()),
    }
)


## 4. Required Evidence Stack

When this notebook is eventually filled in with real results, it should contain the following sections in order.

1. **Frozen methodology recap**
   - one concise restatement of split, horizon, origin, UTC handling, leakage prevention, and official metrics

2. **Final candidate set**
   - final included models only
   - short rationale for why each candidate reached the final stage
   - explicit note on which intermediate experiments are excluded from the final decision table

3. **Validation selection table**
   - the one table that determines the winner under the frozen rule
   - clear indication of the selected model and the runner-up

4. **Why the selected model wins**
   - concise reasoning using the primary metric, tie-breaks, and practicality

5. **Held-out test results**
   - reveal only after the winner is frozen
   - report `MAE`, `RMSE`, `bias`, `rMAE`, and DM where feasible

6. **Runtime and operational practicality**
   - fit time, predict time, warning counts, and whether the model is realistically maintainable

7. **Objective case-week comparison**
   - use the frozen week-selection logic
   - show how the selected model behaves on the typical winter, typical summer, and high-volatility weeks

8. **Limitations and thesis conclusion**
   - what the selected model does well
   - where it still struggles
   - why it is still the best defensible choice under the current scope


## 5. Reporting Style In This Notebook

This notebook should combine two reporting styles.

### Project-standard benchmark style
- concise final comparison tables
- metrics by validation and test
- DM tests
- runtime and practical practicality

### Thesis-interpretive style
- short restatement of how the candidate set was reduced
- why the selected model is not only numerically strong but also methodologically defensible
- if `FS4` is eventually included, retain a short Huang-style interpretive bridge instead of dumping the entire preparatory analysis again

This notebook should be readable as the one document a thesis reader opens when asking: "What is the final answer, and how was that answer chosen fairly?"


In [ ]:
FINAL_NOTEBOOK_TODOS = [
    "Resolve the final candidate run set after all methodology is frozen.",
    "Load only saved artifacts; do not rerun benchmark suites from this notebook.",
    "Construct the final validation selection table using the frozen rule.",
    "Write a one-paragraph decision statement naming the selected model and why it wins.",
    "Reveal and interpret the final held-out test table only after the decision statement is frozen.",
    "Add runtime and case-week evidence for the selected model against the strongest alternatives.",
    "Finish with a thesis-ready conclusion and limitations section.",
]

FINAL_EXPECTED_OUTPUTS = [
    "final_candidate_set_table",
    "validation_model_selection_table",
    "selected_model_decision_log",
    "held_out_test_results_table",
    "diebold_mariano_summary",
    "runtime_practicality_table",
    "objective_case_week_plots",
    "final_conclusion_block",
]

pd.DataFrame(
    {
        "todo": FINAL_NOTEBOOK_TODOS,
    }
)
